# Banking Transaction Analytics
## 1. Data Understanding

### Objective
Explore the structure and quality of the banking
transaction dataset before further analysis.

### Data Source
IBM Synthetic Financial Transactions Dataset

### Scope
- Initial review of 10,000 records
- Column names and data types
- Missing values and duplicate records

### Limitations
Findings from the initial 10,000 rows may not
represent the full dataset.

In [4]:
#Library

import pandas as pd
import os

In [5]:

print("Current Directory:", os.getcwd())
print("CSV Exists:", os.path.exists("../data/raw/HI-Small_Trans.csv"))

Current Directory: c:\Users\Dell\Downloads\banking-transaction-analytics\notebooks
CSV Exists: True


In [6]:
# Load csv file
df = pd.read_csv("../data/raw/HI-Small_Trans.csv",nrows=10000)

print(f"Rows: {df.shape[0]:,}")     #Rows
print(f"Columns: {df.shape[1]}")    #Columns

display(df.info())                           #Data type of each columns
df.head()

Rows: 10,000
Columns: 11
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Timestamp           10000 non-null  object 
 1   From Bank           10000 non-null  int64  
 2   Account             10000 non-null  object 
 3   To Bank             10000 non-null  int64  
 4   Account.1           10000 non-null  object 
 5   Amount Received     10000 non-null  float64
 6   Receiving Currency  10000 non-null  object 
 7   Amount Paid         10000 non-null  float64
 8   Payment Currency    10000 non-null  object 
 9   Payment Format      10000 non-null  object 
 10  Is Laundering       10000 non-null  int64  
dtypes: float64(2), int64(3), object(6)
memory usage: 859.5+ KB


None

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


### Initial Observations

- The sample contains 10,000 transactions.
- The dataset includes 11 columns.
- Some data types may require conversion.
- Further validation is needed before analysis.


## 2. Data Quality Assessment

### 2.1 Missing Values

**Objective:** Assess missing values across all columns.

**Method:** Count null values using Pandas.

In [7]:
# Count missing values in each column
missing_values = df.isnull().sum()

# Calculate missing value percentage for each column
missing_percentage = (missing_values / len(df)) * 100

# Display results
missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage (%)": missing_percentage.round(2)
})

display(missing_summary)


,Missing Values,Missing Percentage (%)
Timestamp,0,0.0
From Bank,0,0.0
Account,0,0.0
To Bank,0,0.0
Account.1,0,0.0
Amount Received,0,0.0
Receiving Currency,0,0.0
Amount Paid,0,0.0
Payment Currency,0,0.0
Payment Format,0,0.0





#### Findings
- **Result:** No missing values in the 10,000-row sample.
- **Impact:** No treatment required for this sample.
- **Status:** Initial check completed.





### 2.2 Duplicate Records

**Objective:** Identify fully duplicated transaction records.

**Method:** Count duplicates across all columns using Pandas.



In [8]:
# Count duplicate records
duplicate_count = df.duplicated().sum()

# Calculate duplicate percentage
duplicate_percentage = (duplicate_count / len(df)) * 100

# Display results
print(f"Duplicate Records: {duplicate_count}")
print(f"Duplicate Percentage: {duplicate_percentage:.2f}%")

Duplicate Records: 0
Duplicate Percentage: 0.00%



#### Findings

- **Result:** No fully duplicated records were detected in the 10,000-row sample (0.00%).
- **Impact:** No duplicate removal is required for the current sample.
- **Status:** Initial check completed.



### 2.3 Transaction Amount Validation

**Objective:** Identify potentially invalid transaction amounts in the dataset.

**Method:** Examine the `Amount Paid` and `Amount Received` columns using Pandas to identify negative values, zero values, and minimum and maximum transaction amounts.


In [9]:
# Select transaction amount columns
amount_columns = ["Amount Paid", "Amount Received"]
#display(df[amount_columns])

# Check negative, zero, minimum, and maximum values
for col in amount_columns:
    negative_count = (df[col] < 0).sum()
    zero_count = (df[col] == 0).sum()
    min_amount = df[col].min()
    max_amount = df[col].max()

    print(f"\n{col}")
    print(f"Negative Values: {negative_count}")
    print(f"Zero Values: {zero_count}")
    print(f"Minimum Amount: {min_amount:,.2f}")
    print(f"Maximum Amount: {max_amount:,.2f}")


Amount Paid
Negative Values: 0
Zero Values: 0
Minimum Amount: 0.01
Maximum Amount: 1,329,257,627.00

Amount Received
Negative Values: 0
Zero Values: 0
Minimum Amount: 0.01
Maximum Amount: 1,329,257,627.00




#### Findings

- **Result:** No negative or zero transaction amounts were detected in the initial 10,000-row sample. Both amount columns have a minimum of 0.01 and a maximum of 1,329,257,627.00.
- **Impact:** No immediate treatment is required for negative or zero amounts. However, unusually large transactions may affect statistical analysis.
- **Status:** Initial validation completed.




### 2.4 High-Value Transaction Investigation

**Objective:** Investigate unusually large transactions and assess whether they are outliers compared to similar transactions.

**Method:** Filter cheque transactions in USD and compare transaction amounts using mean, median, and maximum values.


In [10]:
# Identify the transaction with the highest Amount Paid
highest_transaction = df.loc[df["Amount Paid"].idxmax()]

display(highest_transaction)

# Filter Cheque transactions in USD
cheque_usd = df[
    (df["Payment Format"] == "Cheque") &
    (df["Payment Currency"] == "US Dollar")
]

# Calculate transaction statistics
cheque_count = len(cheque_usd)
mean_amount = cheque_usd["Amount Paid"].mean()
median_amount = cheque_usd["Amount Paid"].median()
max_amount = cheque_usd["Amount Paid"].max()

# Compare the highest transaction with the median
median_ratio = max_amount / median_amount if median_amount > 0 else None

# Display results
print(f"Total Cheque Transactions (USD): {cheque_count:,}")
print(f"Average Amount: ${mean_amount:,.2f}")
print(f"Median Amount: ${median_amount:,.2f}")
print(f"Maximum Amount: ${max_amount:,.2f}")

if median_ratio is not None:
    print(f"Maximum / Median Ratio: {median_ratio:,.2f}x")
else:
    print("Maximum / Median Ratio: N/A")

Timestamp             2022/09/01 00:28
From Bank                         1412
Account                      800CD5E00
To Bank                           1412
Account.1                    800CD6CA0
Amount Received           1329257627.0
Receiving Currency           US Dollar
Amount Paid               1329257627.0
Payment Currency             US Dollar
Payment Format                  Cheque
Is Laundering                        0
Name: 7672, dtype: object

Total Cheque Transactions (USD): 947
Average Amount: $5,898,811.37
Median Amount: $15,470.89
Maximum Amount: $1,329,257,627.00
Maximum / Median Ratio: 85,919.92x


#### Findings

- **Result:** Among 947 USD cheque transactions, the average amount was approximately $5.9 million, while the median was only $15,470.89.
- **Observation:** The largest cheque transaction was approximately 85,920 times the median amount, indicating an extreme difference between the highest transaction value and a typical transaction. This suggests the presence of unusually large transactions that require further investigation.
- **Impact:** The large difference between the mean and median suggests a highly right-skewed distribution, with extremely large transactions increasing the average.
- **Status:** Further investigation required.


### 2.5 Timestamp Validation

**Objective:** Validate transaction timestamps to ensure accurate time-based analysis.

**Method:** Convert the Timestamp column to datetime, identify invalid values, and examine the earliest and latest transaction dates.


In [11]:

# Convert Timestamp to datetime format
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    format="%Y/%m/%d %H:%M",
    errors="coerce"
)

# Check invalid timestamps
invalid_count = df["Timestamp"].isna().sum()

# Identify earliest and latest transactions
earliest_date = df["Timestamp"].min()
latest_date = df["Timestamp"].max()      

# Display results
print(f"Invalid Timestamps: {invalid_count}")
print(f"Earliest Transaction: {earliest_date}")
print(f"Latest Transaction: {latest_date}")
print(f"Data Type: {df['Timestamp'].dtype}")

Invalid Timestamps: 0
Earliest Transaction: 2022-09-01 00:00:00
Latest Transaction: 2022-09-01 00:29:00
Data Type: datetime64[ns]



#### Findings

- **Result:** No invalid timestamps were detected in the initial 10,000-row sample.
- **Date Range:** September 1, 2022, from 00:00 to 00:29.
- **Impact:** The sample covers only 30 minutes, making it insufficient for daily, weekly, or monthly trend analysis.
- **Status:** Initial validation completed.



### 2.6 Dataset Expansion

**Objective:** Expand the dataset from 10,000 to 100,000 records to explore a broader range of banking transactions.

**Method:** Load the first 100,000 records and compare the date range with the initial sample.


In [12]:
# Load a larger dataset for exploratory analysis
df_100k = pd.read_csv("../data/raw/HI-Small_Trans.csv",nrows=100000)

# Verify the dataset size
print(f"Total Rows: {len(df_100k):,}")
print(f"Total Columns: {df_100k.shape[1]}")

Total Rows: 100,000
Total Columns: 11


In [13]:
# Convert Timestamp to datetime
df_100k["Timestamp"] = pd.to_datetime(
    df_100k["Timestamp"],
    format="%Y/%m/%d %H:%M",
    errors="coerce"
)

# Check the date range
print("Earliest Transaction:", df_100k["Timestamp"].min())
print("Latest Transaction:", df_100k["Timestamp"].max())
print("Invalid Timestamps:", df_100k["Timestamp"].isna().sum())


Earliest Transaction: 2022-09-01 00:00:00
Latest Transaction: 2022-09-01 00:29:00
Invalid Timestamps: 0


In [14]:
# Recheck missing values
print("Missing Values:")
display(df_100k.isnull().sum())

# Recheck duplicate records
print("Duplicate Records:", df_100k.duplicated().sum())

# Recheck negative and zero amounts
for col in ["Amount Paid", "Amount Received"]:
    print(f"\n{col}")
    print("Negative Values:", (df_100k[col] < 0).sum())
    print("Zero Values:", (df_100k[col] == 0).sum())


Missing Values:


Timestamp             0
From Bank             0
Account               0
To Bank               0
Account.1             0
Amount Received       0
Receiving Currency    0
Amount Paid           0
Payment Currency      0
Payment Format        0
Is Laundering         0
dtype: int64

Duplicate Records: 0

Amount Paid
Negative Values: 0
Zero Values: 0

Amount Received
Negative Values: 0
Zero Values: 0


In [15]:
df_eda = df_100k.copy()


#### Findings

- **Result:** No duplicate records, negative transaction amounts, or zero-value transactions were detected in the expanded 100,000-row sample.
- **Data Quality:** No missing values were observed in the columns shown in the validation output.
- **Impact:** The initial checks did not identify issues requiring immediate removal of transaction records.
- **Status:** Initial data quality checks completed.


In [16]:
# Verify EDA dataset
print("Dataset Shape:", df_eda.shape)
print("Earliest Transaction:", df_eda["Timestamp"].min())
print("Latest Transaction:", df_eda["Timestamp"].max())
print("Missing Values:", df_eda.isna().sum().sum())

Dataset Shape: (100000, 11)
Earliest Transaction: 2022-09-01 00:00:00
Latest Transaction: 2022-09-01 00:29:00
Missing Values: 0


In [17]:

# Count transactions by timestamp
timestamp_counts = (
    df_eda["Timestamp"]
    .value_counts()
    .sort_index()
)

# Display the first and last timestamps
print("First 5 timestamps:")
display(timestamp_counts.head())

print("Last 5 timestamps:")
display(timestamp_counts.tail())

# Check number of unique timestamps
print("Unique Timestamps:", df_eda["Timestamp"].nunique())


First 5 timestamps:


Timestamp
2022-09-01 00:00:00    3355
2022-09-01 00:01:00    3298
2022-09-01 00:02:00    3271
2022-09-01 00:03:00    3320
2022-09-01 00:04:00    3454
Name: count, dtype: int64

Last 5 timestamps:


Timestamp
2022-09-01 00:25:00    3371
2022-09-01 00:26:00    3407
2022-09-01 00:27:00    3302
2022-09-01 00:28:00    3392
2022-09-01 00:29:00    3273
Name: count, dtype: int64

Unique Timestamps: 30
